# Contextual Experience Replay (CER) | Agent Memory System

In [1]:
# Contextual Experience Replay
# In an agent loop: execute -> record experience -> replay top-K similar -> use for in-context learning
from dataclasses import dataclass, field
from typing import List
from datetime import datetime, timedelta
import math

In [2]:
@dataclass
class Experience:
    task_type: str
    context: str
    action: str
    outcome: str
    success: bool
    quality: float
    timestamp: datetime = field(default_factory=datetime.now)

class CERBuffer:
    def __init__(self, max_size: int = 1000):
        self.buffer: List[Experience] = []
        self.max_size = max_size

    def store(self, exp: Experience):
        self.buffer.append(exp)
        if len(self.buffer) > self.max_size:
            self.buffer.sort(key=lambda e: e.quality, reverse=True)
            self.buffer = self.buffer[:self.max_size]

    def replay(self, task_type: str, k: int = 3) -> List[Experience]:
        def priority(exp: Experience) -> float:
            relevance = 1.0 if exp.task_type == task_type else 0.3
            age_days = (datetime.now() - exp.timestamp).total_seconds() / 86400
            recency = math.exp(-0.1 * age_days)
            failure_bonus = 0.2 if not exp.success else 0.0
            return relevance * 0.4 + recency * 0.2 + exp.quality * 0.3 + failure_bonus * 0.1
        scored = sorted(self.buffer, key=priority, reverse=True)
        return scored[:k]

In [3]:
cer = CERBuffer()
cer.store(Experience("api_debug", "500 errors after deploy", "Checked logs, found missing env var",
                     "Resolved in 15min", True, 0.9, datetime.now() - timedelta(days=2)))
cer.store(Experience("api_debug", "Latency spike after DB migration", "Assumed network issue (wrong)",
                     "Fixed after 3hrs", True, 0.4, datetime.now() - timedelta(days=5)))
cer.store(Experience("deployment", "Deploying new feature", "Used canary deployment",
                     "Zero-downtime, caught memory leak early", True, 0.95, datetime.now() - timedelta(days=1)))

for exp in cer.replay("api_debug", k=2):
    print(f"  {exp.task_type}: {exp.context} -> {exp.action} (quality: {exp.quality})")

  api_debug: 500 errors after deploy -> Checked logs, found missing env var (quality: 0.9)
  api_debug: Latency spike after DB migration -> Assumed network issue (wrong) (quality: 0.4)
